## LIBRARIES AND DECLARATION

In [1]:
import pandas as pd
import json
import re
from IPython.display import FileLink

## CONFIG

In [2]:
STAGE = 3  # Stage 1: 3 attempts | Stage 2: 2 attempts | Stage 3: 2 attempts

CONFIG = {
    (1,): {
        "files": [
            "/kaggle/input/private-dataset/step2_logic1_llama-3.3-70b-versatile_1.csv",
            "/kaggle/input/private-dataset/step2_logic1_llama-3.3-70b-versatile_2.csv",
            "/kaggle/input/private-dataset/step2_logic1_llama-3.3-70b-versatile_3.csv",
        ],
        "out": "step1_classify1_logic1_llama.csv",
    },
    (2, 3): { 
        "files": [
            "/kaggle/input/datasets/dhuyent/stage3/step2_filter2_reference1_gpt-5.6-sol_1.csv",
            "/kaggle/input/datasets/dhuyent/stage3/step2_filter2_reference1_gpt-5.6-sol_2.csv",
        ],
        "out": "classify3_reference1_gpt-5.6-sol.csv",
    },
}

def get_config(stage):
    for keys, cfg in CONFIG.items():
        if stage in keys:
            return cfg
    raise KeyError(f"No config for stage {stage}")

CFG         = get_config(STAGE)
FILES       = CFG["files"]
OUT_PATH    = CFG["out"]
N_ATTEMPTS  = len(FILES)  # số lần chạy = số file
print(f"STAGE {STAGE}: {N_ATTEMPTS} attempts -> {OUT_PATH}")

STAGE 3: 2 attempts -> classify3_reference1_gpt-5.6-sol.csv


## LOAD DATA

In [3]:
df = pd.concat([pd.read_csv(f) for f in FILES], ignore_index=False)
print("Number of ids:", df.shape[0])
print("Unique ids:", df["id"].nunique())
df.head()

Number of ids: 70
Unique ids: 35


,topic,level,id,problem,solution,url,buggy_submission,bug_description,gt_status,gt_input,...,gt_expected_output,gt_reason,filtered_label,llm_model,prompt_strategy,pred_status,pred_input,pred_actual_output,pred_expected_output,pred_reason
0,String,Medium,18,"The string ""PAYPALISHIRING"" is written in a zi...","#pragma GCC optimize(""Ofast"")\n\n#include <ios...",https://leetcode.com/problems/zigzag-conversion,"#pragma GCC optimize(""Ofast"")\n\n#include <ios...",[Reference – Off-by-one index] Changed the mai...,Accepted,NaN,...,NaN,NaN,3,gpt-5.6-sol,few-shot,Runtime Error,"""s = \""AB\"", numRows = 2""",NaN,"""AB""","The loop condition uses i <= s.size(), so the ..."
1,String,Medium,20,The count-and-say sequence is a sequence of di...,class Solution {\npublic:\n string countAnd...,https://leetcode.com/problems/count-and-say,class Solution {\npublic:\n string countAnd...,[Reference – Off-by-one index] Changed the loo...,Wrong Answer,n =\n11,...,"""11131221133112132113212221""",NaN,2,gpt-5.6-sol,few-shot,Runtime Error,"""n = 2""",NaN,"""11""",The loop executes with i == s.size() and acces...
2,String,Medium,29,"Given an array of strings strs, group the anag...",class Solution {\n public:\n vector<vector<st...,https://leetcode.com/problems/group-anagrams,class Solution {\n public:\n vector<vector<st...,[Reference – Off-by-one index] Changed the sor...,Wrong Answer,"strs = \n[""cab"",""tin"",""pew"",""duh"",""may"",""ill"",...",...,"[[""max""],[""buy""],[""doc""],[""may""],[""ill""],[""duh...",NaN,3,gpt-5.6-sol,few-shot,Runtime Error,"""strs = [\""a\""]""",NaN,"""[[\""a\""]]""","Calling sort(begin(key), end(key) + 1) creates..."
3,String,Hard,35,"Given a string s, return whether s is a valid ...",class Solution {\npublic:\n bool isNumber(s...,https://leetcode.com/problems/valid-number,class Solution {\npublic:\n bool isNumber(s...,[Reference – Off-by-one index] Changed the dig...,Accepted,NaN,...,NaN,NaN,3,gpt-5.6-sol,few-shot,Runtime Error,"""s = \""0\""""",NaN,"""true""",strip_int uses i <= s.size() and therefore acc...
4,String,Hard,46,A transformation sequence from word beginWord ...,class Solution {\n public:\n vector<vector<st...,https://leetcode.com/problems/word-ladder-ii,class Solution {\n public:\n vector<vector<st...,[Reference – Off-by-one index] Changed the cha...,Time Limit Exceeded,NaN,...,NaN,"Last Executed Input\nbeginWord =\n""aaaaa""\nend...",3,gpt-5.6-sol,few-shot,Runtime Error,"""beginWord = \""a\"", endWord = \""b\"", wordList ...",NaN,"""[[\""a\"",\""b\""]]""",getChildren iterates with i <= s.length() and ...


## LABEL CLASSIFICATION

Classify each `id` based on how well `pred_status` (LLM prediction) matches `gt_status` (ground truth) across multiple runs (attempts).

- **Label 1**: ALL attempts match (`pred_status == gt_status`)
- **Label 2**: SOME but not all attempts match (partial match)
- **Label 3**: NO attempt matches

In [4]:
# So sánh gt_status vs pred_status
df["match"] = df.apply(
    lambda r: "sim" if r["gt_status"] == r["pred_status"] else "diff",
    axis=1
)

# Group theo id, đếm sim - diff trong mỗi nhóm
group_stats = (
    df.groupby("id")["match"]
    .value_counts()
    .unstack(fill_value=0)
    .rename(columns={"sim": "n_sim", "diff": "n_diff"})
    .reset_index()
)

# Đảm bảo cả hai cột luôn tồn tại dù tất cả là sim hoặc diff
for col in ("n_sim", "n_diff"):
    if col not in group_stats.columns:
        group_stats[col] = 0

In [5]:
# Check if ids whose `pred_status` differs on EVERY attempt
uniq = df.groupby("id")["pred_status"].nunique().reset_index(name="n_unique")
cnt  = df.groupby("id").size().reset_index(name="n_rows")
uniq = uniq.merge(cnt, on="id")

ids_all_diff = uniq.loc[
    (uniq["n_rows"] == N_ATTEMPTS) & (uniq["n_unique"] == N_ATTEMPTS),
    "id",
]

print("Number of ids with a different pred_status on every attempt:", len(ids_all_diff))
print(ids_all_diff.tolist())

Number of ids with a different pred_status on every attempt: 5
[73, 112, 117, 139, 140]


In [6]:
def assign_label(row, n_attempts=N_ATTEMPTS):
    s, d = row["n_sim"], row["n_diff"]
    total = s + d
    if total != n_attempts:
        return None          # group has missing/extra rows: flag for inspection
    if s == total:           # all sim
        return 1
    if d == total:           # all diff
        return 3
    return 2                 # partial match

group_stats["label"] = group_stats.apply(assign_label, axis=1)

df = df.drop(columns=["label"], errors="ignore")
df = df.merge(group_stats[["id", "label"]], on="id", how="left")

print(df[["id", "gt_status", "pred_status", "match", "label"]].head(20))
print()
print("Label distribution (per id):")
print(df.drop_duplicates("id")["label"].value_counts(dropna=False).sort_index())

     id            gt_status    pred_status match  label
0    18             Accepted  Runtime Error  diff      3
1    20         Wrong Answer  Runtime Error  diff      3
2    29         Wrong Answer  Runtime Error  diff      3
3    35             Accepted  Runtime Error  diff      3
4    46  Time Limit Exceeded  Runtime Error  diff      3
5    53         Wrong Answer   Wrong Answer   sim      1
6    57             Accepted   Wrong Answer  diff      3
7    60         Wrong Answer   Wrong Answer   sim      1
8    61         Wrong Answer  Runtime Error  diff      3
9    63         Wrong Answer   Wrong Answer   sim      1
10   71             Accepted   Wrong Answer  diff      3
11   73         Wrong Answer   Wrong Answer   sim      2
12   76         Wrong Answer  Runtime Error  diff      3
13   78         Wrong Answer  Runtime Error  diff      3
14   80  Time Limit Exceeded  Runtime Error  diff      3
15   81         Wrong Answer  Runtime Error  diff      3
16   96  Time Limit Exceeded  R

## SAVE OUTPUT

In [7]:
df_out = df.drop_duplicates("id").reset_index(drop=True)
df_out.to_csv(OUT_PATH, index=False)
print("Saved", df_out.shape[0], "ids to", OUT_PATH)

Saved 35 ids to classify3_reference1_gpt-5.6-sol.csv


In [8]:
FileLink(OUT_PATH)

/kaggle/working/classify3_reference1_gpt-5.6-sol.csv